# Check the recorded step counts of the February re-run objects

The objects shared on November 6, 2026 (`2026_11_06_qc_regen_nick_andy_coords/`) come with `step_counts.tsv`: one row
per sample with the reads, cells or nuclei at every step. This notebook recounts every number from its primary source,
not from the script that wrote the table, and compares the two.

| column | recounted from |
|---|---|
| final nuclei, positioned (Andy), positioned (Nick) | the shared objects (`qc_passed_{mouse,human}.h5ad`) |
| pass QC thresholds, doublets removed | each sample's QC log (`2026_11_06_qc_regen/logs/01_qc_<sample>.log`) |
| CellBender nuclei | the barcodes in each CellBender output (`xmens_inputs_regen/<sample>/`) |
| Cell Ranger cells, reads | each lane's Cell Ranger output (`velocity/<lane>/cellranger/outs/`) |

Everything is read-only. Run it with the rsc_uv environment's Python; the checks take a few seconds.

In [ ]:
from pathlib import Path
import re

import h5py
import pandas as pd
from anndata.io import read_elem

NH = Path("/n/lab_storage/eddy_lab/users/nhilgert")
NS = Path("/n/netscratch/eddy_lab/Everyone/nhilgert")

SHARED = NS / "2026_11_06_qc_regen_nick_andy_coords"  # the shared objects and their step_counts.tsv
QC_LOGS = NS / "2026_11_06_qc_regen" / "logs"  # logs of the QC run that made the objects
CELLBENDER = NH / "decidualization" / "xmens_inputs_regen"  # July 2026 CellBender run
CELLRANGER = NH / "decidualization" / "revisions" / "velocity"  # February 2026 Cell Ranger run

COUNT_COLUMNS = [
    "reads",
    "Cell Ranger cells",
    "CellBender nuclei",
    "pass QC thresholds",
    "doublets removed",
    "final nuclei",
    "positioned (Andy)",
    "positioned (Nick)",
]

## The recorded table

In [ ]:
recorded = pd.read_csv(SHARED / "step_counts.tsv", sep="\t")
recorded_samples = recorded[recorded["sample"] != "total"].set_index("sample")
recorded_totals = recorded[recorded["sample"] == "total"].set_index("species")
recorded

## Final and positioned nuclei, from the shared objects

A nucleus counts as positioned when both of its coordinates are present. Only `obs` is read, so the count matrices
stay on disk.

In [ ]:
def count_nuclei(path):
    """Nuclei per sample in one shared object, and how many each coordinate set places."""
    with h5py.File(path, "r") as f:
        obs = read_elem(f["obs"])

    placed_andy = obs["x_um_dbscan_andy"].notna() & obs["y_um_dbscan_andy"].notna()
    placed_nick = obs["x_um_dbscan_nick"].notna() & obs["y_um_dbscan_nick"].notna()
    return pd.DataFrame({
        "final nuclei": obs.groupby("sample", observed=True).size(),
        "positioned (Andy)": placed_andy.groupby(obs["sample"], observed=True).sum(),
        "positioned (Nick)": placed_nick.groupby(obs["sample"], observed=True).sum(),
    })


from_objects = pd.concat(
    [count_nuclei(SHARED / f"qc_passed_{species}.h5ad") for species in ("mouse", "human")]
)
from_objects

## Threshold and doublet counts, from the QC logs

Each sample's log records the nuclei CellBender passed on, how many passed the thresholds, how many DoubletDetection
flagged, and how many remained.

In [ ]:
def read_qc_log(sample):
    """The QC log's counts for one sample."""
    text = (QC_LOGS / f"01_qc_{sample}.log").read_text()
    return {
        "CellBender nuclei (log)": int(re.search(r"loaded (\d+) cells", text).group(1)),
        "pass QC thresholds": int(re.search(r"after threshold QC: (\d+)/", text).group(1)),
        "doublets removed": int(re.search(r"flagged (\d+) doublets", text).group(1)),
        "final nuclei (log)": int(re.search(r"final: (\d+) cells", text).group(1)),
    }


from_logs = pd.DataFrame({sample: read_qc_log(sample) for sample in recorded_samples.index}).T
from_logs

## CellBender nuclei, from the CellBender outputs

In [ ]:
def count_cellbender_nuclei(sample):
    """Barcodes CellBender called as cells: the columns of its filtered matrix."""
    with h5py.File(CELLBENDER / sample / f"{sample}_cellbender_filtered.h5", "r") as f:
        return len(f["matrix/barcodes"])


from_cellbender = pd.Series(
    {sample: count_cellbender_nuclei(sample) for sample in recorded_samples.index},
    name="CellBender nuclei",
)
from_cellbender

## Cell Ranger cells and reads, from the Cell Ranger outputs

Cell Ranger cells are the barcodes in each lane's filtered matrix. `metrics_summary.csv` reports the read count, and
the cell count again as Estimated Number of Cells.

In [ ]:
def read_cellranger(lane):
    """Reads and called cells for one lane."""
    outs = CELLRANGER / lane / "cellranger" / "outs"
    metrics = pd.read_csv(outs / "metrics_summary.csv", thousands=",")
    with h5py.File(outs / "filtered_feature_bc_matrix.h5", "r") as f:
        n_barcodes = len(f["matrix/barcodes"])
    return {
        "reads": int(metrics["Number of Reads"][0]),
        "Cell Ranger cells": n_barcodes,
        "estimated cells (metrics)": int(metrics["Estimated Number of Cells"][0]),
    }


from_cellranger = pd.DataFrame(
    {sample: read_cellranger(lane) for sample, lane in recorded_samples["lane"].items()}
).T
from_cellranger

## Compare

Where two sources report the same number, they must agree first. Then every recounted number is compared with the
recorded table, sample by sample and for the species totals.

In [ ]:
recounted = pd.concat(
    [from_cellranger, from_cellbender, from_logs, from_objects], axis=1
).loc[recorded_samples.index]

consistency = {
    "CellBender output = QC log 'loaded'": (
        recounted["CellBender nuclei"] == recounted["CellBender nuclei (log)"]
    ),
    "object nuclei = QC log 'final'": recounted["final nuclei"] == recounted["final nuclei (log)"],
    "thresholds - doublets = final": (
        recounted["pass QC thresholds"] - recounted["doublets removed"] == recounted["final nuclei"]
    ),
    "filtered barcodes = estimated cells": (
        recounted["Cell Ranger cells"] == recounted["estimated cells (metrics)"]
    ),
}
for check, agrees in consistency.items():
    print(f"{check}: {int(agrees.sum())} of {len(agrees)} samples")

In [ ]:
per_sample = recounted[COUNT_COLUMNS].astype(int)
differs = per_sample != recorded_samples[COUNT_COLUMNS]
print(f"per sample: {int((~differs).values.sum())} of {differs.size} numbers match the recorded table")

totals = per_sample.groupby(recorded_samples["species"]).sum()
totals_differ = totals != recorded_totals.loc[totals.index, COUNT_COLUMNS]
print(f"species totals: {int((~totals_differ).values.sum())} of {totals_differ.size} numbers match")

mismatched = differs.any(axis=1)
if mismatched.any():
    side_by_side = pd.concat(
        {"recorded": recorded_samples[COUNT_COLUMNS], "recounted": per_sample}, axis=1
    )
    print(side_by_side[mismatched])
else:
    print("no differences")